In [ ]:
# Install required packages
%pip install databricks-sdk --upgrade mlflow --quiet
dbutils.library.restartPython()

In [ ]:
# DBTITLE 1,Setup Widget Parameters
# Create widgets for parameterized execution
dbutils.widgets.text("catalog", "analytics_uc", "Catalog Name")
dbutils.widgets.text("schema", "ml_models", "Schema Name")
dbutils.widgets.text("model_name", "", "Model Name")
dbutils.widgets.text("endpoint_name", "", "Endpoint Name")
dbutils.widgets.dropdown("workload_size", "Small", ["Small", "Medium", "Large"], "Workload Size")
dbutils.widgets.dropdown("scale_to_zero", "true", ["true", "false"], "Scale to Zero")
dbutils.widgets.text("model_version", "", "Model Version (optional)")
dbutils.widgets.text("secret_scope", "ml-secrets", "Secret Scope")
dbutils.widgets.text("secret_key", "databricks-token", "Secret Key")

# Optional tags (comma-separated key:value pairs)
dbutils.widgets.text("tags", "", "Tags (format: key1:value1,key2:value2)")

In [ ]:
# DBTITLE 1,Get Parameters
# Retrieve widget parameters
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
model_name = dbutils.widgets.get("model_name")
endpoint_name = dbutils.widgets.get("endpoint_name")
workload_size = dbutils.widgets.get("workload_size")
scale_to_zero = dbutils.widgets.get("scale_to_zero").lower() == "true"
model_version = dbutils.widgets.get("model_version") or None
secret_scope = dbutils.widgets.get("secret_scope")
secret_key = dbutils.widgets.get("secret_key")
tags_str = dbutils.widgets.get("tags")

# Validate required parameters
if not model_name:
    raise ValueError("model_name parameter is required")

if not endpoint_name:
    # Default to model name with -api suffix
    endpoint_name = f"{model_name}-api"
    print(f"ℹ️  No endpoint_name specified, using: {endpoint_name}")

# Parse tags if provided
tags = {}
if tags_str:
    try:
        for tag_pair in tags_str.split(","):
            key, value = tag_pair.split(":")
            tags[key.strip()] = value.strip()
    except:
        print(f"⚠️  Warning: Could not parse tags: {tags_str}")
        tags = {}

# Display configuration
print("=" * 80)
print("DEPLOYMENT CONFIGURATION")
print("=" * 80)
print(f"Full Model Name: {catalog}.{schema}.{model_name}")
print(f"Endpoint Name: {endpoint_name}")
print(f"Workload Size: {workload_size}")
print(f"Scale to Zero: {scale_to_zero}")
print(f"Model Version: {model_version or 'Latest'}")
if tags:
    print(f"Custom Tags: {tags}")
print("=" * 80)

In [ ]:
# DBTITLE 1,Import Deployment Module
import sys
import os

# Add src directory to path
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
src_path = os.path.join(os.path.dirname(os.path.dirname(notebook_path)), "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from deployment.deploy_endpoint import deploy_model_endpoint, ModelServingDeploymentError

print(f"✅ Deployment module imported successfully")

In [ ]:
# DBTITLE 1,Get Databricks Token from Secrets
try:
    token = dbutils.secrets.get(scope=secret_scope, key=secret_key)
    print(f"✅ Retrieved token from secrets: {secret_scope}/{secret_key}")
except Exception as e:
    print(f"❌ Failed to retrieve token from secrets: {e}")
    print(f"   Scope: {secret_scope}")
    print(f"   Key: {secret_key}")
    raise

In [ ]:
# DBTITLE 1,Deploy Model to Serving Endpoint
print("\n🚀 Starting model deployment...")
print("-" * 80)

try:
    # Add default tags
    deployment_tags = {
        "deployed_by": "workflow",
        "deployment_notebook": notebook_path
    }

    # Add custom tags
    if tags:
        deployment_tags.update(tags)

    # Deploy the model
    result = deploy_model_endpoint(
        token=token,
        catalog=catalog,
        schema=schema,
        model_name=model_name,
        endpoint_name=endpoint_name,
        workload_size=workload_size,
        scale_to_zero_enabled=scale_to_zero,
        model_version=model_version,
        tags=deployment_tags
    )

    print("\n" + "=" * 80)
    print("✅ DEPLOYMENT SUCCESSFUL!")
    print("=" * 80)
    print(f"📍 Endpoint URL: {result['endpoint_url']}")
    print(f"📦 Model: {result['full_model_name']}")
    print(f"🔢 Version: {result['model_version']}")
    print(f"🏷️  Endpoint: {result['endpoint_name']}")
    print("=" * 80)

    # Export results for downstream tasks
    dbutils.jobs.taskValues.set("endpoint_url", result['endpoint_url'])
    dbutils.jobs.taskValues.set("endpoint_name", result['endpoint_name'])
    dbutils.jobs.taskValues.set("deployed_model_version", result['model_version'])
    dbutils.jobs.taskValues.set("full_model_name", result['full_model_name'])

    # Exit with success
    dbutils.notebook.exit(result['endpoint_url'])

except ModelServingDeploymentError as e:
    print("\n" + "=" * 80)
    print("❌ DEPLOYMENT FAILED")
    print("=" * 80)
    print(f"Error: {e}")
    print("=" * 80)
    raise

except Exception as e:
    print("\n" + "=" * 80)
    print("❌ UNEXPECTED ERROR")
    print("=" * 80)
    print(f"Error: {e}")
    print("=" * 80)
    import traceback
    traceback.print_exc()
    raise

## Verify Deployment

The cells below can be used to verify the deployment was successful.

In [ ]:
# DBTITLE 1,Check Endpoint Status (Optional)
from deployment.deploy_endpoint import get_endpoint_status

try:
    status = get_endpoint_status(endpoint_name, token)

    print("\n📊 Endpoint Status Check:")
    print("-" * 80)
    print(f"Name: {status['name']}")
    print(f"State: {status['state']}")
    print(f"Creator: {status['creator']}")
    print(f"Created: {status['creation_timestamp']}")

    if status.get('tags'):
        print("\nTags:")
        for key, value in status['tags'].items():
            print(f"  {key}: {value}")

    print("-" * 80)

except Exception as e:
    print(f"⚠️  Could not retrieve endpoint status: {e}")

## Usage in Workflows

Add this as a task in your workflow YAML:

```yaml
- task_key: deploy_model_serving
  depends_on:
    - task_key: train_model
  notebook_task:
    notebook_path: ../src/deployment/deploy_model
    base_parameters:
      catalog: "analytics_uc"
      schema: "ml_models"
      model_name: "customer_churn_predictor"
      endpoint_name: "churn-prediction-api"
      workload_size: "Small"
      scale_to_zero: "true"
      secret_scope: "ml-secrets"
      secret_key: "databricks-token"
      tags: "environment:production,team:ml"
  existing_cluster_id: ${var.cluster_id}
```

## Manual Execution

To run this notebook manually in Databricks:

1. Navigate to the notebook
2. Fill in the widget parameters at the top
3. Click "Run All"

## Retrieving Task Values in Downstream Tasks

Access deployment results in downstream workflow tasks:

```python
# Get values from previous task
endpoint_url = dbutils.jobs.taskValues.get(
    taskKey="deploy_model_serving",
    key="endpoint_url"
)

deployed_version = dbutils.jobs.taskValues.get(
    taskKey="deploy_model_serving",
    key="deployed_model_version"
)

print(f"Testing endpoint: {endpoint_url}")
print(f"Deployed version: {deployed_version}")
```